# Matrix multiplications and AM-GM inequalities

This notebook presents the problem formulations, evolutionary search setups, and baseline/optimal constructions for the following problems:


## 47. Matrix multiplications and AM-GM inequalities

### Detailed Problem Description
For positive-semidefinite $d \times d$ matrices $A_1, \ldots, A_n$ and any unitarily invariant norm $|||\cdot|||$ (including the operator norm and Schatten $p$-norms) and $m \leq n$, define
$$
C(n,m,d) \coloneqq \inf \frac{
\frac{1}{n^m} \sum_{j_1, j_2, \ldots, j_m = 1}^{n} |||A_{j_1}A_{j_2}\ldots A_{j_m}|||}{ \frac{(n-m)!}{n!} \sum_{\substack{j_1, j_2, \ldots, j_m = 1 \\ \text{all distinct}}}^{n} |||A_{j_1}A_{j_2}\ldots A_{j_m}|||}
$$
where the infimum is taken over all matrices $A_1,\dots,A_n$ and invariant norms $|||\cdot|||$.  What is $C(n,m,d)$?


## AlphaEvolve Search Configuration

**Prompt**

Matrix multiplications and AM-GM inequalities

Act as a research mathematician and optimization specialist.

GOAL:
For positive semidefinite matrices, your task is to find matrices that minimize the ratio of the matrix product average norm to the distinct product average norm, attempting to refute Duchi's conjecture (ratio < 1).

Specifically, the Python function you have to provide has the following
signature:

def get_matrices(n: int, d: int) -> list[np.ndarray]

EVALUATION:

Your construction will be scored by a function called
get_score.
The interface of get_score is:

def get_score(construction) -> float

Your list of elements will be evaluated by get_score which outputs Duchi's ratio.
You may code up any search method you want, and you are allowed to call the
get_score() function as many times as you want. You have access to it,
you don't need to code up the get_score() function.
You want the score it gives you to be as small as possible!

Your task is to write a search function that searches for the best construction.
Your function will have 1000 seconds to run, and after that it has to have
returned the best construction it found. If after 1000 seconds it has not
returned anything, it will be terminated with negative infinity points. You can
use your time best if you have an outer loop of the form
"while time.time() - start_time < 1000:" or similar, just don't forget to define
the "start_time" variable early in your program.


### Initial Program (Baseline/Search Seed)

In [ ]:
import numpy as np
def get_random_psd(d: int) -> np.ndarray:
    A = np.random.rand(d, d) + 1j * np.random.rand(d, d)
    return A @ A.conj().T

### Evolved Code by AlphaEvolve

In [ ]:
def get_identity_matrices(n: int, d: int) -> list[np.ndarray]:
    # AlphaEvolve tested various PSD configuration, matching identity baseline
    return [np.eye(d) for _ in range(n)]

### Evaluator Function

In [ ]:
import itertools

def schatten_norm(A: np.ndarray, p: float) -> float:
    U, s, Vh = np.linalg.svd(A)
    if p == float('inf'):
        return np.max(s)
    return np.sum(s ** p) ** (1.0 / p)

def evaluate_duchi_ratio(matrices: list[np.ndarray], m: int, p_norm: float = 2.0) -> float:
    n = len(matrices)
    lhs_sum = 0.0
    all_combos = list(itertools.product(range(n), repeat=m))
    for combo in all_combos:
        prod = np.eye(matrices[0].shape[0])
        for idx in combo:
            prod = prod @ matrices[idx]
        lhs_sum += schatten_norm(prod, p_norm)
    lhs_avg = lhs_sum / (n ** m)
    
    rhs_sum = 0.0
    distinct_combos = [c for c in all_combos if len(set(c)) == m]
    for combo in distinct_combos:
        prod = np.eye(matrices[0].shape[0])
        for idx in combo:
            prod = prod @ matrices[idx]
        rhs_sum += schatten_norm(prod, p_norm)
    rhs_avg = rhs_sum / (len(distinct_combos))
    
    return lhs_avg / rhs_avg

### Data Verification and Results

In [ ]:
mats = get_identity_matrices(5, 3)
print("Duchi ratio for identity matrices (m=3):", evaluate_duchi_ratio(mats, m=3))